# Slider: 가능한 개선 전부 검증 (Colab)

실험 세 가지를 차례로 돌리고, 모두 **수당 30ms 시간 제한 + 수 제한 없음(1000수)** 대국에서 **SPRT** 로 판정합니다.
기준은 튜닝한 선형 가중치(`weights/tuned.json`)와 기본 탐색입니다.

| 실험 | 내용 | 예상 시간 (코어 2개) |
|---|---|---|
| A. 탐색 개선 | PVS, LMR, 널무브를 하나씩 켜고 기본 탐색과 대국. 통과한 것이 2개 이상이면 합친 것도 대국 | 1시간 안팎 |
| B. 가중치 재튜닝 | 수 제한 없는 자가대전 600판으로 Texel 튜닝을 다시 하고, 새 가중치 vs 지금 가중치 | 1시간 안팎 |
| C. 신경망 재확인 | B 의 데이터로 특징 입력 신경망을 학습해 깊이 4 고정 / 시간 제한 대국 | 40분 안팎 |

- 런타임은 **CPU** (GPU 안 씀). 위에서부터 차례로 실행하세요.
- 결과는 단계마다 Google Drive `slider_results/all_experiments/state.json` 에 저장됩니다. 세션이 끊기면 처음부터 다시 실행해도 **끝난 대국과 데이터는 건너뜁니다.**
- SPRT 판정: `H1` = Elo 25 이상 강함(채택), `H0` = 그만큼 강하지 않음, `undecided` = 최대 판 수까지 결론 없음.
- 오프닝 북은 넣지 않았습니다. 비교 대국이 무작위 시작 국면에서 시작해서 북의 효과를 잴 수 없기 때문입니다.

## 1. 설정

In [ ]:
import os, json, shutil, subprocess, pathlib, time
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/Perdume/Slider.git"
BRANCH = "rust-port"

# ----- 공통 대국 설정 -----
MATCH_TIME_MS = 30            # 수당 시간 (ms)
SPRT_ELO0, SPRT_ELO1 = 0, 25  # H0: Elo 0 이하 / H1: Elo 25 이상
SPRT_MAX_GAMES = 800          # SPRT 최대 판 수 (결론이 나면 일찍 끝남)
MAX_PLIES = 1000              # 수 제한 (사실상 없음. 같은 국면 3번이면 무승부)
SEED = 1

# ----- 실험 A: 탐색 개선 -----
RUN_SEARCH = True
SEARCH_TESTS = ["pvs", "lmr", "nmp"]

# ----- 실험 B: 가중치 재튜닝 -----
RUN_TUNE = True
TUNE_GAMES = 600
TUNE_TIME_MS = 30
DRAW_VALUE = 0.5   # 튜닝 목표에서 무승부의 값. 0.0 = "둘 다 진 것" 이지만, 로지스틱 모형이 상수항(tempo)만 크게 바꿀 수 있어서 기본은 0.5

# ----- 실험 C: 신경망 재확인 -----
RUN_NN = True
HIDDEN = 32
NN_EPOCHS = 60
NN_DEPTH = 4

SAVE_TO_DRIVE = True
THREADS = os.cpu_count() or 2
WORK = pathlib.Path("/content/slider_all"); WORK.mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = str(pathlib.Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")   # 로그인 창이 뜨면 허용
    OUT = pathlib.Path("/content/drive/MyDrive/slider_results/all_experiments")
else:
    OUT = WORK / "out"
(OUT / "data").mkdir(parents=True, exist_ok=True)
STATE_PATH = OUT / "state.json"
print("threads:", THREADS, "| 결과 저장:", OUT)

## 2. 도우미 함수 (실행, 진행 상황 저장, SPRT 대국)

In [ ]:
def sh(cmd, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, cwd=cwd, env=env)


def run(binary, args):
    cmd = [str(binary), *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        line = line.rstrip("\n"); lines.append(line); print("   ", line, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"실행 실패 (exit {p.returncode}): {cmd[0]}")
    return lines


state = json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else {}


def save_state():
    STATE_PATH.write_text(json.dumps(state, ensure_ascii=False, indent=1), encoding="utf-8")


def fmt(r):
    if "aWins" not in r:
        return str(r)
    return (f"{r['aWins']}승 {r['draws']}무 {r['bWins']}패 ({r['games']}판), Elo {r['elo']:+.0f} "
            f"({r['eloLow']:+.0f} ~ {r['eloHigh']:+.0f}), SPRT {r.get('sprt', '-')}")


def sprt(key, a_args, b_args, limit=None, note=""):
    """A 쪽 인자, B 쪽 인자로 SPRT 대국. 결과는 state[key] 에 저장되고, 이미 있으면 건너뜁니다."""
    if key in state:
        print(f"[{key}] 이미 끝남: {fmt(state[key])}")
        return state[key]
    limit = limit or ["--time-ms", MATCH_TIME_MS]
    lines = run(BIN / "matchup", [*a_args, *b_args, "--games", SPRT_MAX_GAMES, *limit, "--max-plies", MAX_PLIES,
                                  "--sprt-elo0", SPRT_ELO0, "--sprt-elo1", SPRT_ELO1, "--threads", THREADS, "--seed", SEED + 1000])
    r = json.loads([l for l in lines if l.startswith("RESULT ")][-1][len("RESULT "):])
    r["note"] = note
    state[key] = r
    save_state()
    print(f"[{key}] {fmt(r)}")
    return r


def cached_file(name):
    """WORK 에 없으면 Drive 사본을 가져온다. 있으면 경로, 없으면 None."""
    local = WORK / name
    if not (local.exists() and local.stat().st_size > 0):
        remote = OUT / "data" / name
        if remote.exists() and remote.stat().st_size > 0:
            shutil.copy(remote, local)
            print("Drive 에서 가져옴:", remote)
    return local if local.exists() and local.stat().st_size > 0 else None


def keep_file(path):
    shutil.copy(path, OUT / "data" / path.name)

## 3. Rust 설치, 코드 받기, 빌드

In [ ]:
if shutil.which("cargo") is None:
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
REPO = WORK / "Slider"
if REPO.exists():
    sh(["git", "-C", REPO, "pull", "--ff-only"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO])
sh(["git", "-C", REPO, "log", "--oneline", "-1"])

TARGET = WORK / f"target_{HIDDEN}"
sh(["cargo", "build", "--release", "--bin", "selfplay", "--bin", "trainnet", "--bin", "matchup"],
   cwd=REPO / "rust", env={**os.environ, "SLIDER_HIDDEN": str(HIDDEN), "CARGO_TARGET_DIR": str(TARGET)})
BIN = TARGET / "release"
W = REPO / "weights" / "tuned.json"   # 기준 가중치
print("기준 가중치:", W.read_text())

## 실험 A. 탐색 개선 (PVS / LMR / 널무브)
각 기법을 켠 엔진(A) vs 기본 엔진(B). 가중치는 같습니다.

In [ ]:
if RUN_SEARCH:
    for o in SEARCH_TESTS:
        sprt(f"search_{o}", ["--a", W, "--opts-a", o], ["--b", W], note=f"탐색 {o} vs 기본")
    passed = [o for o in SEARCH_TESTS if state.get(f"search_{o}", {}).get("sprt") == "H1"]
    if len(passed) >= 2:
        combo = ",".join(passed)
        sprt(f"search_{combo}", ["--a", W, "--opts-a", combo], ["--b", W], note=f"탐색 {combo} vs 기본")
    print("\n통과한 탐색 개선:", passed or "없음")
else:
    print("RUN_SEARCH = False")

## 실험 B-1. 수 제한 없는 자가대전 데이터 (기준 가중치)
C 의 신경망 학습에도 같은 데이터를 씁니다.

In [ ]:
tune_name = f"tune_{TUNE_GAMES}g_{TUNE_TIME_MS}ms_p{MAX_PLIES}.jsonl"
tune_data = cached_file(tune_name)
if (RUN_TUNE or RUN_NN) and tune_data is None:
    t0 = time.time()
    tune_data = WORK / tune_name
    run(BIN / "selfplay", ["--games", TUNE_GAMES, "--time-ms", TUNE_TIME_MS, "--random-plies", 4, "--epsilon", 0.05,
                           "--max-plies", MAX_PLIES, "--threads", THREADS, "--seed", SEED, "--weights", W,
                           "--positions", "yes", "--out", tune_data])
    keep_file(tune_data)
    print(f"자가대전 {time.time() - t0:.0f}초")
print("데이터:", tune_data)

## 실험 B-2. Texel 튜닝 함수 (기존 튜닝 노트북과 같은 방식)

In [ ]:
from scipy.optimize import minimize

# EvalWeights.java 의 NAMES / 기본값과 같아야 합니다 (데이터를 읽을 때 길이를 확인합니다)
NAMES = ["threat1", "threat2", "threat2PerThreat", "mobility", "reach",
         "plan2", "plan3", "plan4", "plan5", "position", "tempo"]
DEFAULT_WEIGHTS = dict(zip(NAMES, [-150, -400, -50, 1, 2, 150, 70, 30, 12, 1, 0]))
FORCED_SCORE = 50_000  # 이 이상이면 탐색이 승패를 확정한 국면
LAMBDAS = [0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
REG_UNIT = 200.0       # 정규화에서 가중치 차이를 재는 단위
MAX_ABS_WEIGHT = 3000


def write_weights(path, w):
    with open(path, "w", encoding="utf-8") as f:
        json.dump({k: int(w[k]) for k in NAMES}, f, indent=2)


def to_vector(w):
    return np.array([w[k] for k in NAMES], dtype=float)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def mse(w, X, y, scale):
    return float(np.mean((sigmoid(X @ w / scale) - y) ** 2))


K_MIN, K_MAX = 20.0, 20000.0


def calibrate_scale(w, X, y):
    """가중치 w 를 고정하고, P(승) = sigmoid(점수 / K) 의 K 를 오차가 가장 작은 값으로 고른다."""
    grid = np.exp(np.linspace(np.log(K_MIN), np.log(K_MAX), 300))
    errs = [mse(w, X, y, k) for k in grid]
    return float(grid[int(np.argmin(errs))])


def load_samples(files):
    """files = [(라운드, jsonl 경로), ...]  ->  X(특징), y(결과), gid(판 번호; 라운드가 달라도 겹치지 않음)"""
    frames = []
    for rnd, path in files:
        df = pd.read_json(path, lines=True)
        df["gid"] = rnd * 1_000_000 + df["game"]
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    X = np.array(df["features"].tolist(), dtype=float)
    if X.shape[1] != len(NAMES):
        raise ValueError(f"특징이 {X.shape[1]}개인데 NAMES 는 {len(NAMES)}개입니다. EvalWeights.java 와 맞춰 주세요.")
    keep = (df["score"].abs() < FORCED_SCORE).values
    print(f"국면 {len(df):,}개 중 확정 승패 {(~keep).sum():,}개 제외 -> {keep.sum():,}개 사용 "
          f"(판 {df['gid'].nunique()}개, 승 {np.mean(df['outcome'] == 1):.1%} / 무 {np.mean(df['outcome'] == 0.5):.1%} / 패 {np.mean(df['outcome'] == 0):.1%})")
    return (X[keep], df["outcome"].values[keep].astype(float), df["gid"].values[keep],
            df["score"].values[keep].astype(float))


def corr(a, b):
    if np.std(a) < 1e-9 or np.std(b) < 1e-9:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def diagnose(X, y, t, w0):
    """데이터에 학습할 만한 신호가 있는지 보여 준다. 상관계수가 0 에 가까우면 가중치를 바꿔도 얻을 게 적다."""
    ev = X @ w0
    print(f"신호 세기 (상관계수): 평가점수-승패 {corr(ev, y):+.3f} | 탐색점수-승패 {corr(t, y):+.3f} | 평가점수-탐색점수 {corr(ev, t):+.3f}")
    print(f"점수 표준편차: 평가점수 {ev.std():.0f} / 탐색점수 {t.std():.0f}  (평가가 훨씬 크면 평가가 탐색보다 과하게 확신한다는 뜻)")


def fit(X, y, w0, lam, scale):
    """제곱오차 + lam * |(w - w0) / REG_UNIT|^2 를 최소화하는 가중치. 눈금 scale 은 고정."""
    s = X.std(axis=0)
    s[s < 1e-9] = 1.0              # 상수 열 (tempo 등) 은 그대로 둔다
    Xs = X / s                     # 열 크기를 맞춘다

    # 변수 v = w * s / scale 로 바꿔서 기울기가 O(0.01~1) 이 되게 한다.
    # (w 그대로 쓰면 기울기가 1e-5 쯤이라 L-BFGS 가 시작하자마자 "수렴"으로 착각하고 멈춘다)
    def loss(v):
        w = v * scale / s
        p = sigmoid(Xs @ v)
        err = p - y
        diff = (w - w0) / REG_UNIT
        value = np.mean(err ** 2) + lam * np.sum(diff ** 2)
        grad = (2.0 / len(y)) * (Xs.T @ (err * p * (1.0 - p))) + 2.0 * lam * diff * scale / (REG_UNIT * s)
        return value, grad

    res = minimize(loss, w0 * s / scale, jac=True, method="L-BFGS-B",
                   options={"maxiter": 2000, "gtol": 1e-10, "ftol": 1e-14})
    return res.x * scale / s


def tune(X, y, gid, w0, seed=0):
    """눈금 K 를 보정하고 lam 을 검증 오차로 고른 뒤, 전체 데이터로 다시 맞춘 가중치(dict)를 돌려준다.
    신호가 너무 약해서 K 를 정할 수 없으면 None."""
    rng = np.random.default_rng(seed)
    games = np.unique(gid)
    rng.shuffle(games)
    val = np.isin(gid, games[: max(1, int(len(games) * 0.2))])
    tr = ~val

    k_train = calibrate_scale(w0, X[tr], y[tr])
    print(f"눈금 K = {k_train:.0f}  (현재 가중치 기준. 클수록 '점수가 승패를 덜 말해 준다'는 뜻)")
    if k_train > 0.8 * K_MAX:
        print("!! K 가 탐색 범위 끝에 붙었습니다. 현재 평가 점수가 자가대전 승패를 거의 설명하지 못한다는 뜻이라,")
        print("   가중치 크기가 임의로 정해져 학습 결과를 믿을 수 없습니다. 이 라운드는 학습을 건너뜁니다.")
        print("   (데이터는 쌓이므로 다음 라운드에 합쳐서 다시 시도합니다. GAMES_PER_ROUND / TIME_MS 를 늘리면 도움이 됩니다.)")
        return None
    print(f"검증 오차 기준선: 항상 0.5 예측 {np.mean((0.5 - y[val]) ** 2):.4f} | "
          f"현재 가중치 (K=200 가정) {mse(w0, X[val], y[val], 200.0):.4f} | 현재 가중치 (보정한 K) {mse(w0, X[val], y[val], k_train):.4f}")
    rows = []
    for lam in LAMBDAS:
        w = fit(X[tr], y[tr], w0, lam, k_train)
        rows.append({"lambda": lam, "train_mse": mse(w, X[tr], y[tr], k_train), "val_mse": mse(w, X[val], y[val], k_train)})
    table = pd.DataFrame(rows)
    print(table.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    best_lam = float(table.loc[table["val_mse"].idxmin(), "lambda"])
    print("선택한 lambda:", best_lam)

    k_all = calibrate_scale(w0, X, y)
    w = fit(X, y, w0, best_lam, k_all)
    w = np.clip(np.round(w), -MAX_ABS_WEIGHT, MAX_ABS_WEIGHT)
    return dict(zip(NAMES, w.astype(int).tolist()))

## 실험 B-3. 재튜닝 + 새 가중치 vs 기준 가중치

In [ ]:
if RUN_TUNE:
    new_path = WORK / "weights_retuned.json"
    base = json.loads(W.read_text())
    if cached_file(new_path.name) is None:
        X, y, gid, t = load_samples([(1, tune_data)])
        y = np.where(y == 0.5, DRAW_VALUE, y)
        w0 = to_vector(base)
        diagnose(X, y, t, w0)
        new = tune(X, y, gid, w0, seed=SEED)
        if new is None:
            state["retune"] = {"note": "K 를 정할 수 없어 튜닝 건너뜀"}; save_state()
        else:
            write_weights(new_path, new)
            keep_file(new_path)
    if new_path.exists():
        new = json.loads(new_path.read_text())
        print(pd.DataFrame({"기준": base, "재튜닝": new}).assign(변화=lambda d: d["재튜닝"] - d["기준"]).to_string())
        if new == base:
            print("재튜닝 결과가 기준과 같아서 대국을 건너뜁니다")
        else:
            sprt("retune", ["--a", new_path], ["--b", W], note="재튜닝 가중치 vs 기준 가중치")
else:
    print("RUN_TUNE = False")

## 실험 C. 특징 입력 신경망 (기준 가중치 + 신경망 vs 기준 가중치)

In [ ]:
if RUN_NN:
    net = WORK / f"net_feat_{HIDDEN}_{tune_data.stem}.txt"
    if cached_file(net.name) is None:
        lines = run(BIN / "trainnet", ["--data", tune_data, "--weights", W, "--out", net, "--epochs", NN_EPOCHS,
                                       "--lambda", 1.0, "--features", "yes", "--seed", SEED])
        best = [l for l in lines if "가장 좋은 검증 손실" in l]
        state["nn_train"] = {"note": best[-1].strip() if best else ""}; save_state()
        keep_file(net)
    sprt("nn_depth", ["--a", W, "--net-a", net], ["--b", W], limit=["--time-ms", 100000, "--depth", NN_DEPTH],
         note=f"신경망, 깊이 {NN_DEPTH} 고정")
    sprt("nn_time", ["--a", W, "--net-a", net], ["--b", W], note=f"신경망, 수당 {MATCH_TIME_MS}ms")
else:
    print("RUN_NN = False")

## 요약 (Drive 의 `all_experiments/summary.txt` 에도 저장)

In [ ]:
lines = [f"설정: 수당 {MATCH_TIME_MS}ms, SPRT [{SPRT_ELO0}, {SPRT_ELO1}], 최대 {SPRT_MAX_GAMES}판, 수 제한 {MAX_PLIES}, "
         f"튜닝 데이터 {TUNE_GAMES}판 x {TUNE_TIME_MS}ms, 무승부 값 {DRAW_VALUE}, 은닉 {HIDDEN}"]
for key, r in state.items():
    lines.append(f"{key:<22} {r.get('note', ''):<28} {fmt(r) if 'aWins' in r else ''}")
text = "\n".join(lines)
print(text)
(OUT / "summary.txt").write_text(text + "\n\nJSON:\n" + json.dumps(state, ensure_ascii=False, indent=1), encoding="utf-8")
print("\n저장:", OUT / "summary.txt")